In [28]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error
import matplotlib.pyplot as plt

Connect SQLite3 to pandas to get data.

In [29]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('../../market_data.db')

In [30]:
sales_df = pd.read_sql("SELECT * FROM sales_history", conn)
competitor_df = pd.read_sql("SELECT * FROM competitor_intelligence", conn)
inventory_df = pd.read_sql("SELECT * FROM inventory_cost", conn)

In [31]:
sales_df.head()
competitor_df.head()
inventory_df.head()



,product_id,current_stock,base_cost
0,WIDGET-A,100,10.0
1,GADGET-B,50,25.0
2,SENSOR-C,200,5.0


Merge all three tables and Start of Day-5


In [32]:
df=sales_df.merge(
    competitor_df,
    on='product_id',
    how='left'
).merge(
    inventory_df,
    on='product_id',
    how='left'
)
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost
0,1,2024-01-01,WIDGET-A,12,47.25,1,1,44.48,2024-01-01,100,10.0
1,1,2024-01-01,WIDGET-A,12,47.25,1,2,48.01,2024-01-02,100,10.0
2,1,2024-01-01,WIDGET-A,12,47.25,1,3,42.81,2024-01-03,100,10.0
3,1,2024-01-01,WIDGET-A,12,47.25,1,4,48.90,2024-01-04,100,10.0
4,1,2024-01-01,WIDGET-A,12,47.25,1,5,41.83,2024-01-05,100,10.0


Sort the values by product ID and Date.

In [33]:
df['date']=pd.to_datetime(df['date'])
df=df.sort_values(by=['product_id','date'])
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost
8100,91,2024-01-01,GADGET-B,17,28.45,0,91,28.99,2024-01-01,50,25.0
8101,91,2024-01-01,GADGET-B,17,28.45,0,92,33.13,2024-01-02,50,25.0
8102,91,2024-01-01,GADGET-B,17,28.45,0,93,29.01,2024-01-03,50,25.0
8103,91,2024-01-01,GADGET-B,17,28.45,0,94,25.06,2024-01-04,50,25.0
8104,91,2024-01-01,GADGET-B,17,28.45,0,95,26.59,2024-01-05,50,25.0


New Column Price Gap and Price Gap Percentage.


In [34]:
df['price_gap']=df['price']-df['competitor_price']
df['price_gap_percentage']=round((df['price']-df['competitor_price'])/df['competitor_price']*100,2)
df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost,price_gap,price_gap_percentage
8100,91,2024-01-01,GADGET-B,17,28.45,0,91,28.99,2024-01-01,50,25.0,-0.54,-1.86
8101,91,2024-01-01,GADGET-B,17,28.45,0,92,33.13,2024-01-02,50,25.0,-4.68,-14.13
8102,91,2024-01-01,GADGET-B,17,28.45,0,93,29.01,2024-01-03,50,25.0,-0.56,-1.93
8103,91,2024-01-01,GADGET-B,17,28.45,0,94,25.06,2024-01-04,50,25.0,3.39,13.53
8104,91,2024-01-01,GADGET-B,17,28.45,0,95,26.59,2024-01-05,50,25.0,1.86,7.00


New Column : Inventory Velocity

In [35]:
# 1. Calculate a 7-day rolling average of units sold per product
df['rolling_units_7d'] = df.groupby('product_id')['units_sold'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# 2. Calculate Velocity: How much of our current stock do we sell daily (on average)?
# We use a small epsilon (0.0001) or fillna to avoid division by zero errors
df['inventory_velocity'] = df['rolling_units_7d'] / df['current_stock'].replace(0, 1)

# 3. Round for cleanliness
df['inventory_velocity'] = df['inventory_velocity'].round(3)

df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost,price_gap,price_gap_percentage,rolling_units_7d,inventory_velocity
8100,91,2024-01-01,GADGET-B,17,28.45,0,91,28.99,2024-01-01,50,25.0,-0.54,-1.86,17.0,0.34
8101,91,2024-01-01,GADGET-B,17,28.45,0,92,33.13,2024-01-02,50,25.0,-4.68,-14.13,17.0,0.34
8102,91,2024-01-01,GADGET-B,17,28.45,0,93,29.01,2024-01-03,50,25.0,-0.56,-1.93,17.0,0.34
8103,91,2024-01-01,GADGET-B,17,28.45,0,94,25.06,2024-01-04,50,25.0,3.39,13.53,17.0,0.34
8104,91,2024-01-01,GADGET-B,17,28.45,0,95,26.59,2024-01-05,50,25.0,1.86,7.00,17.0,0.34


New Column: Price Elasticity


In [36]:
# 1. Calculate percentage change in quantity and price
df['pct_change_q'] = df.groupby('product_id')['units_sold'].pct_change()
df['pct_change_p'] = df.groupby('product_id')['price'].pct_change()

# 2. Elasticity = (% Change in Q) / (% Change in P)
df['price_elasticity'] = df['pct_change_q'] / df['pct_change_p']

# 3. Clean up: replace infinite values (from 0 price change) with 0
import numpy as np
df['price_elasticity'] = df['price_elasticity'].replace([np.inf, -np.inf], 0).fillna(0)

df.head()

,sale_id,date,product_id,units_sold,price,promotion_status,intel_id,competitor_price,timestamp,current_stock,base_cost,price_gap,price_gap_percentage,rolling_units_7d,inventory_velocity,pct_change_q,pct_change_p,price_elasticity
8100,91,2024-01-01,GADGET-B,17,28.45,0,91,28.99,2024-01-01,50,25.0,-0.54,-1.86,17.0,0.34,NaN,NaN,0.0
8101,91,2024-01-01,GADGET-B,17,28.45,0,92,33.13,2024-01-02,50,25.0,-4.68,-14.13,17.0,0.34,0.0,0.0,0.0
8102,91,2024-01-01,GADGET-B,17,28.45,0,93,29.01,2024-01-03,50,25.0,-0.56,-1.93,17.0,0.34,0.0,0.0,0.0
8103,91,2024-01-01,GADGET-B,17,28.45,0,94,25.06,2024-01-04,50,25.0,3.39,13.53,17.0,0.34,0.0,0.0,0.0
8104,91,2024-01-01,GADGET-B,17,28.45,0,95,26.59,2024-01-05,50,25.0,1.86,7.00,17.0,0.34,0.0,0.0,0.0


ML: Linear Regression 
Some Important Answers:
If a simple linear regression performs worse than random, it indicates a fundamental issue in the pipeline — either the data quality is poor, the features do not encode the underlying demand logic, or the target variable is incorrectly constructed (often due to leakage or misalignment).
In such a case, adding a more complex model would only amplify noise, not improve performance.
A random train-test split is dangerous because this is time-series data. Randomly mixing past and future observations causes data leakage, allowing the model to indirectly see future demand patterns during training. A time-based split preserves causality by training on earlier dates and validating on later dates.


Column Name: Demand Trend

In [37]:
# Calculate the % change in the 7-day rolling average
# This tells us if the 'pace' of sales is accelerating or slowing down
df['demand_trend'] = df.groupby('product_id')['rolling_units_7d'].pct_change()

# Clean up: Replace NaN (from the first row) and any division-by-zero errors with 0
df['demand_trend'] = df['demand_trend'].fillna(0).replace([float('inf'), float('-inf')], 0)

print("Column 'demand_trend' successfully added.")

Column 'demand_trend' successfully added.


ML: Features and Prediction

In [38]:
df["units_sold_next_7_days"] = (
    df.groupby("product_id")["units_sold"]
      .rolling(window=7, min_periods=7)
      .sum()
      .shift(-7)
      .reset_index(level=0, drop=True)
)

In [39]:
feature_cols=[
    'price',
    'price_elasticity',
    'inventory_velocity',
    'price_gap',
    'demand_trend'
]
df_clean=df.dropna(subset=feature_cols + ['units_sold_next_7_days'])
X = df_clean[feature_cols]
y = df_clean['units_sold_next_7_days']
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Columns in X:", X.columns.tolist())


X shape: (24281, 5)
y shape: (24281,)
Columns in X: ['price', 'price_elasticity', 'inventory_velocity', 'price_gap', 'demand_trend']


Phase-2 Day-6: Baseline Model Sanity Check

In [40]:
print(f"Rows in X: {len(X)}")
print(f"Rows in y: {len(y)}")

# If those are 0, check your source:
# print(df.head())

Rows in X: 24281
Rows in y: 24281


In [51]:
print("Sales rows:", len(sales_df))
print("Competitor rows:", len(competitor_df))
df_clean = df_clean.drop_duplicates(subset=['product_id','date'])

print(df_clean.duplicated().sum())
df_clean.groupby(['product_id','date']).size().sort_values(ascending=False).head(10)

Sales rows: 270
Competitor rows: 270
0


product_id  date      
WIDGET-A    2024-03-30    1
GADGET-B    2024-01-01    1
            2024-01-02    1
            2024-01-03    1
            2024-01-04    1
            2024-01-05    1
            2024-01-06    1
            2024-01-07    1
            2024-01-08    1
            2024-01-09    1
dtype: int64

In [52]:
split_index = int(len(df_clean) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

model=LinearRegression()
model.fit(X_train,y_train)

y_pred=model.predict(X_test)

print(f"MSE : {mean_absolute_error(y_test,y_pred):.2f}")
print(f"R2 Score: {r2_score(y_test,y_pred):.2f}")

MSE : 31.84
R2 Score: -1.43


In [55]:
df_clean[['price','price_elasticity','inventory_velocity','price_gap','demand_trend']].describe()

df_clean[['price',
          'price_elasticity',
          'inventory_velocity',
          'price_gap',
          'demand_trend',
          'units_sold_next_7_days']].corr()

,price,price_elasticity,inventory_velocity,price_gap,demand_trend,units_sold_next_7_days
price,1.000000,-0.002946,-0.776718,0.648343,-0.108184,-0.896784
price_elasticity,-0.002946,1.000000,-0.045592,0.064123,0.047033,0.028885
inventory_velocity,-0.776718,-0.045592,1.000000,-0.585045,-0.120155,0.720240
price_gap,0.648343,0.064123,-0.585045,1.000000,-0.326607,-0.600005
demand_trend,-0.108184,0.047033,-0.120155,-0.326607,1.000000,0.340799
units_sold_next_7_days,-0.896784,0.028885,0.720240,-0.600005,0.340799,1.000000
